In [1]:
# ======================================================================
# DUBLIN BER (processed): keep only Dublin homes and add location columns
# ======================================================================
# Input:  the 19 national processed BER tables
#         (dataSet\processed_dataset\housing_energy_dataset)
# Output: the same 19 tables, Dublin only
#         (just_dublin_dataset\just_dublin_processed_dataset\ber)
#
# BER writes Dublin as 'Co. Dublin' or a postal district like 'Dublin 12'.
# The location columns are added to the assessment table, and every other
# table keeps only the Dublin homes (matched by ber_id).
#
# Needs 00_dublin_location_lookup.ipynb to be run first.
# ======================================================================

import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)

base = r'C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project'

national_ber = base + r'\dataSet\processed_dataset\housing_energy_dataset'
dublin_processed = base + r'\just_dublin_dataset\just_dublin_processed_dataset'
dublin_ber = dublin_processed + r'\ber'

os.makedirs(dublin_ber, exist_ok=True)

In [2]:
# The BER county table lists every place name BER uses.
# The Dublin ones are 'Co. Dublin' and 'Dublin 1' ... 'Dublin 24'.

ber_county = pd.read_csv(national_ber + r'\Final_datset_county.csv')

dublin_names = ber_county[ber_county['CountyName'].str.contains('Dublin')]
dublin_names

,county_id,CountyName
5,6,Co. Dublin
27,28,Dublin 1
28,29,Dublin 10
29,30,Dublin 11
30,31,Dublin 12
31,32,Dublin 13
32,33,Dublin 14
33,34,Dublin 15
34,35,Dublin 16
35,36,Dublin 17


In [3]:
# Loading the main BER table and keeping only Dublin homes

assessment = pd.read_csv(national_ber + r'\Final_datset_assessment.csv')
print('All of Ireland:', len(assessment))

dublin_ids = list(dublin_names['county_id'])
assessment = assessment[assessment['county_id'].isin(dublin_ids)]
print('Dublin only:   ', len(assessment))

All of Ireland: 569058
Dublin only:    181195


In [4]:
# Putting the BER place name back next to each home ('Co. Dublin', 'Dublin 12')

name_map = dict(zip(ber_county['county_id'], ber_county['CountyName']))
assessment['ber_location'] = assessment['county_id'].map(name_map)

assessment['ber_location'].value_counts()

ber_location
Co. Dublin    48327
Dublin 15     13465
Dublin 18     12075
Dublin 24     11841
Dublin 12      7457
Dublin 8       7260
Dublin 11      7119
Dublin 22      6906
Dublin 9       6903
Dublin 7       6265
Dublin 13      5829
Dublin 4       5789
Dublin 16      5670
Dublin 14      5362
Dublin 3       5120
Dublin 5       5034
Dublin 1       4556
Dublin 6       4106
Dublin 6W      2808
Dublin 10      2631
Dublin 17      2363
Dublin 2       2246
Dublin 20      2063
Name: count, dtype: int64

In [5]:
# District: if the BER place is a postal district, that is the district.
# 'Co. Dublin' homes don't say which part of Dublin they are in, so they
# become 'Dublin (no district)'.

lookup = pd.read_csv(dublin_processed + r'\dublin_location_lookup.csv')
district_list = list(lookup['dublin_district'])

assessment['dublin_district'] = 'Dublin (no district)'
assessment.loc[assessment['ber_location'].isin(district_list), 'dublin_district'] = assessment['ber_location']

assessment['dublin_district'].value_counts()

dublin_district
Dublin (no district)    48327
Dublin 15               13465
Dublin 18               12075
Dublin 24               11841
Dublin 12                7457
Dublin 8                 7260
Dublin 11                7119
Dublin 22                6906
Dublin 9                 6903
Dublin 7                 6265
Dublin 13                5829
Dublin 4                 5789
Dublin 16                5670
Dublin 14                5362
Dublin 3                 5120
Dublin 5                 5034
Dublin 1                 4556
Dublin 6                 4106
Dublin 6W                2808
Dublin 10                2631
Dublin 17                2363
Dublin 2                 2246
Dublin 20                2063
Name: count, dtype: int64

In [6]:
# Adding the other location levels from the lookup (district_id, county,
# county_id, province, nation).
# local_authority stays empty: BER doesn't say which council a home is in,
# and districts cross council borders (e.g. Dublin 6W is split between two),
# so it can't be worked out safely.

assessment = assessment.drop(columns=['county_id'])   # BER's own id, replaced by the lookup ids
assessment = assessment.merge(lookup[['dublin_district', 'district_id', 'county', 'county_id',
                                      'province', 'nation']],
                              on='dublin_district', how='left')

assessment['local_authority'] = np.nan

print('Rows without a district id:', assessment['district_id'].isnull().sum())   # should be 0
assessment[['ber_id', 'ber_location', 'dublin_district', 'district_id', 'county',
            'province', 'nation', 'local_authority']].head()

Rows without a district id: 0


,ber_id,ber_location,dublin_district,district_id,county,province,nation,local_authority
0,14,Dublin 12,Dublin 12,13,Dublin,Leinster,Ireland,NaN
1,19,Dublin 11,Dublin 11,12,Dublin,Leinster,Ireland,NaN
2,24,Dublin 7,Dublin 7,8,Dublin,Leinster,Ireland,NaN
3,25,Dublin 3,Dublin 3,3,Dublin,Leinster,Ireland,NaN
4,27,Dublin 9,Dublin 9,10,Dublin,Leinster,Ireland,NaN


In [7]:
# How many Dublin homes have a real district (not 'no district')?

has_district = (assessment['dublin_district'] != 'Dublin (no district)').mean() * 100
print('Homes with a postal district:', round(has_district, 1), '%')

Homes with a postal district: 73.3 %


In [8]:
# Saving the Dublin assessment table

assessment.to_csv(dublin_ber + r'\dublin_assessment.csv', index=False)
print('Saved dublin_assessment.csv', assessment.shape)

Saved dublin_assessment.csv (181195, 19)


In [9]:
# The other tables: keep only rows for Dublin homes (same ber_id).
# The three small lookup tables (county, dwelling_type, fuel) are copied as they are.

dublin_ber_ids = set(assessment['ber_id'])

other_tables = ['results', 'geometry', 'fabric', 'ventilation', 'space_heating',
                'water_heating', 'solar_water', 'group_heating', 'nzeb_renewables',
                'floors', 'walls', 'other_energy', 'group_boilers', 'solar_pv', 'wind_turbines']

for name in other_tables:
    table = pd.read_csv(national_ber + '\\Final_datset_' + name + '.csv', low_memory=False)
    rows_before = len(table)
    table = table[table['ber_id'].isin(dublin_ber_ids)]
    table.to_csv(dublin_ber + '\\dublin_' + name + '.csv', index=False)
    print(name, ':', rows_before, '->', len(table))

results : 569058 -> 181195
geometry : 569058 -> 181195
fabric : 569058 -> 181195
ventilation : 569058 -> 181195
space_heating : 569058 -> 181195
water_heating : 569058 -> 181195
solar_water : 569058 -> 181195
group_heating : 569058 -> 181195
nzeb_renewables : 569058 -> 181195
floors : 975905 -> 303070
walls : 952212 -> 315354
other_energy : 124817 -> 31085
group_boilers : 37003 -> 34000
solar_pv : 182 -> 29
wind_turbines : 1 -> 0


In [10]:
for name in ['county', 'dwelling_type', 'fuel']:
    table = pd.read_csv(national_ber + '\\Final_datset_' + name + '.csv')
    table.to_csv(dublin_ber + '\\dublin_' + name + '.csv', index=False)
    print('Copied', name, table.shape)

Copied county (52, 2)
Copied dwelling_type (10, 2)
Copied fuel (18, 2)
